In [1]:
import sys
from pathlib import Path
# Production layout: add project root and src (run from repo root or notebooks/ingestion/)
_root = Path(".").resolve()
if _root.name == "ingestion":
    _root = _root.parent.parent
elif (_root / "src").is_dir():
    pass
else:
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "src"))

from storage.postgres.pgConn import PgConn
from storage.postgres import PostgresSQL_table_queries
from storage.cloud.CloudStorage import CloudStorageProvider

import pandas as pd
from datetime import datetime

In [2]:
pg_conn = PgConn("historical")
df = pg_conn.get_stocks_prices()

Connection to the database successful!
Table name set to: historical


In [3]:
df.head()

,ref,book,date,open,high,low,close,adj_close,volume
0,https://finance.yahoo.com,tusd-btc,2023-08-09,0.000034,0.000034,0.000034,0.000034,0.000034,102207
1,https://finance.yahoo.com,tusd-btc,2023-08-08,0.000034,0.000034,0.000034,0.000034,0.000034,74850
2,https://finance.yahoo.com,tusd-btc,2023-08-07,0.000034,0.000034,0.000034,0.000034,0.000034,34264
3,https://finance.yahoo.com,tusd-btc,2023-08-06,0.000034,0.000034,0.000034,0.000034,0.000034,29283
4,https://finance.yahoo.com,tusd-btc,2023-08-05,0.000034,0.000034,0.000034,0.000034,0.000034,78352


In [4]:
df.count()

ref          119179
book         119179
date         119179
open         119179
high         119179
low          119179
close        119179
adj_close    119179
volume       119179
dtype: int64

In [5]:
# Assuming df is your DataFrame
date_column_type = df['date'].dtype
print("Type of values in 'date' column:", date_column_type)

Type of values in 'date' column: object


In [6]:
class DataETL():
    
    def __init__(self, dataframe):
            self.df = dataframe
    
    class Export():
        def __init__(self, dataframe):
            self.cloudProvider = CloudStorage.CloudStorageProvider()
            self.df = dataframe
            
        def export_stocks_to_s3(self, bucket_name, prefix_path, file_format):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()

            # Create a new bucket
            aws_storage.create_bucket(bucket_name)

            # Upload DataFrame with datetime subfolder structure
            aws_storage.upload_dataframe_with_timestamp(self.df, bucket_name, prefix_path, file_format)
            
        def export_stocks_to_s3_full_file(self, bucket_name, prefix_path, filename):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()
            aws_storage.upload_dataframe_to_csv(self.df, bucket_name, filename, prefix_path)
            
    class Ingestion():
        
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorage.CloudStorageProvider()
            
        def get_full_data_csv_file(self, bucket_name, prefix_path):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_csv_from_specific_folder(bucket_name, prefix_path)
    
    class Process():
        
        def __init__(self, dataframe):
            self.df = dataframe
        
        def getData():
            self.df = pg_conn.get_financial_news()
        
        def filter_by_current_date(self):
            # Get the current date as a string
            today_date_str = datetime.today().strftime('%Y-%m-%d')

            # Extract the year, month, and day from the current date string
            current_year, current_month, current_day = today_date_str.split('-')

            # Filter the DataFrame by comparing the substrings of the datetime column
            filtered_df = self.df[self.df['date'].str.startswith(f'{current_year}-{current_month}-{current_day}')]

            return filtered_df
        
        def filter_by_date_range(self, df, start_date, end_date):
            # Convert start_date and end_date strings to datetime objects
            start_date = pd.to_datetime(start_date)
            end_date = pd.to_datetime(end_date)
            
            df['date'] = pd.to_datetime(df['date'])
            # Filter by date range
            filtered_df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]
            return filtered_df
        
        def filter_by_book(self, df, book=None):
            # Filter by book if specified
            if book is not None:
                filtered_df = df[df['book'] == book]
            return filtered_df
        
        def get_unique_by_date(self, df):
            # Sort the DataFrame by 'date' in descending order
            df_sorted = df.sort_values(by='date', ascending=False)

            # Drop duplicates, keeping only the first occurrence for each unique combination of book and date
            df_unique_latest = df_sorted.drop_duplicates(subset=['book', 'date'])
            df_unique_latest.count()
            return df_unique_latest
            
    class Transform():
        def extractStopWords():
            pass

In [7]:
FILTER_BY_CURRENT_DATE = False
FILTER_BY_RANGE_DATE = True
FILTER_BY_BOOK_DATE = False
# TODO 2017-2016
etl = DataETL(df)
etl_process = etl.Process(etl.df)
uniqued_df = etl_process.get_unique_by_date(etl_process.df)
processed_df = uniqued_df

if FILTER_BY_CURRENT_DATE == True:
    processed_df = etl_process.filter_by_current_date(uniqued_df)
    processed_df.head()
elif FILTER_BY_RANGE_DATE == True:
    start_date='2025-11-23'
    end_date='2026-01-25'
    processed_df = etl_process.filter_by_date_range(uniqued_df, start_date, end_date)
    processed_df.head()
elif FILTER_BY_BOOK_DATE == True:
    processed_df = etl_process.filter_by_current_date(uniqued_df)
    processed_df.head()

In [8]:
etl_export = etl.Export(processed_df)
bucket_name = "test-financial-stocks-bucket"
prefix_path = "stocks/crypto"
post_full_csv = False
post_to_s3 = True
now = datetime.now()
filename = f"{now.year}-{now.month:02}-{now.day:02}_full_record"
file_format = "csv"
if post_to_s3 == True:
    etl_export.export_stocks_to_s3(bucket_name, prefix_path, file_format)
if post_full_csv == True:
    etl_export.export_stocks_to_s3_full_file(bucket_name, prefix_path, filename)

Bucket 'test-financial-stocks-bucket' created successfully.
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=snx-usd/year=2026/month=01/day=25/format=csv/20260125-snx-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=dydx-usd/year=2026/month=01/day=25/format=csv/20260125-dydx-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=ada-usd/year=2026/month=01/day=25/format=csv/20260125-ada-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=yfi-usd/year=2026/month=01/day=25/format=csv/20260125-yfi-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=trx-usd/year=2026/month=01/day=25/format=csv/20260125-trx-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=eth-usd/year=2026/month=01/day=25/format=csv/20260125-eth

Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=xlm-usd/year=2026/month=01/day=24/format=csv/20260124-xlm-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=atom-usd/year=2026/month=01/day=24/format=csv/20260124-atom-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=wif-usd/year=2026/month=01/day=24/format=csv/20260124-wif-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=render-usd/year=2026/month=01/day=24/format=csv/20260124-render-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=floki-usd/year=2026/month=01/day=24/format=csv/20260124-floki-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=xrp-usd/year=2026/month=01/day=24/format=csv/20260124-xrp-usd.csv'
Data uploaded to S3 bucket 'test-financi

Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=bat-usd/year=2026/month=01/day=23/format=csv/20260123-bat-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=chz-usd/year=2026/month=01/day=23/format=csv/20260123-chz-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=wif-usd/year=2026/month=01/day=23/format=csv/20260123-wif-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=doge-usd/year=2026/month=01/day=23/format=csv/20260123-doge-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=xrp-usd/year=2026/month=01/day=23/format=csv/20260123-xrp-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=bal-usd/year=2026/month=01/day=23/format=csv/20260123-bal-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-

Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=ltc-usd/year=2026/month=01/day=22/format=csv/20260122-ltc-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=wif-usd/year=2026/month=01/day=22/format=csv/20260122-wif-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=shib-usd/year=2026/month=01/day=22/format=csv/20260122-shib-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=bonk-usd/year=2026/month=01/day=22/format=csv/20260122-bonk-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=enj-usd/year=2026/month=01/day=22/format=csv/20260122-enj-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=dydx-usd/year=2026/month=01/day=22/format=csv/20260122-dydx-usd.csv'
Data uploaded to S3 bucket 'test-financial-sto

Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=near-usd/year=2026/month=01/day=21/format=csv/20260121-near-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=doge-usd/year=2026/month=01/day=21/format=csv/20260121-doge-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=dot-usd/year=2026/month=01/day=21/format=csv/20260121-dot-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=ltc-usd/year=2026/month=01/day=21/format=csv/20260121-ltc-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=link-usd/year=2026/month=01/day=21/format=csv/20260121-link-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=qnt-usd/year=2026/month=01/day=21/format=csv/20260121-qnt-usd.csv'
Data uploaded to S3 bucket 'test-financial-sto

...

Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=tigres-usd/year=2025/month=11/day=27/format=csv/20251127-tigres-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=omg-usd/year=2025/month=11/day=27/format=csv/20251127-omg-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=ada-usd/year=2025/month=11/day=27/format=csv/20251127-ada-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=wif-usd/year=2025/month=11/day=27/format=csv/20251127-wif-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=bar-usd/year=2025/month=11/day=27/format=csv/20251127-bar-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=mana-usd/year=2025/month=11/day=27/format=csv/20251127-mana-usd.csv'
Data uploaded to S3 bucket 'test-financial-s

Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=tigres-usd/year=2025/month=11/day=26/format=csv/20251126-tigres-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=atom-usd/year=2025/month=11/day=26/format=csv/20251126-atom-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=floki-usd/year=2025/month=11/day=26/format=csv/20251126-floki-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=bch-usd/year=2025/month=11/day=26/format=csv/20251126-bch-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=virtual-usd/year=2025/month=11/day=26/format=csv/20251126-virtual-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=dot-usd/year=2025/month=11/day=26/format=csv/20251126-dot-usd.csv'
Data uploaded to S3 bucket 'test

Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=trx-usd/year=2025/month=11/day=25/format=csv/20251125-trx-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=sol-usd/year=2025/month=11/day=25/format=csv/20251125-sol-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=shib-usd/year=2025/month=11/day=25/format=csv/20251125-shib-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=lrc-usd/year=2025/month=11/day=25/format=csv/20251125-lrc-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=tusd-btc/year=2025/month=11/day=25/format=csv/20251125-tusd-btc.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=avax-usd/year=2025/month=11/day=25/format=csv/20251125-avax-usd.csv'
Data uploaded to S3 bucket 'test-financial-sto

Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=algo-usd/year=2025/month=11/day=24/format=csv/20251124-algo-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=axs-usd/year=2025/month=11/day=24/format=csv/20251124-axs-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=paxg-usd/year=2025/month=11/day=24/format=csv/20251124-paxg-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=eth-usd/year=2025/month=11/day=24/format=csv/20251124-eth-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=dot-usd/year=2025/month=11/day=24/format=csv/20251124-dot-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=fet-usd/year=2025/month=11/day=24/format=csv/20251124-fet-usd.csv'
Data uploaded to S3 bucket 'test-financial-stock

Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=paxg-usd/year=2025/month=11/day=23/format=csv/20251123-paxg-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=omg-usd/year=2025/month=11/day=23/format=csv/20251123-omg-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=psg-usd/year=2025/month=11/day=23/format=csv/20251123-psg-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=avax-usd/year=2025/month=11/day=23/format=csv/20251123-avax-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=eth-usd/year=2025/month=11/day=23/format=csv/20251123-eth-usd.csv'
Data uploaded to S3 bucket 'test-financial-stocks-bucket' under folder 'stocks/crypto/book=sushi-usd/year=2025/month=11/day=23/format=csv/20251123-sushi-usd.csv'
Data uploaded to S3 bucket 'test-financial-s

In [9]:
ingest_data = False
get_full_file = True
df_from_file = None

if ingest_data == True:
    etl_ingestion = etl.Ingestion(etl.df)
    bucket_name = "test-financial-stocks-bucket"
    prefix_path = "stocks/crypto/"
    year = ''
    mont = ''
    day = ''
    hour = ''
    minute = ''
    if get_full_file == True:
        df_from_file = etl_ingestion.get_full_data_csv_file(bucket_name, prefix_path)

In [10]:
if df_from_file is not None:
    print(df_from_file.head())